# SEC Financial Statement Data — Cắt tỉa & Làm sạch dữ liệu

**Dự án**: Phân tích tài chính 50 công ty lớn (5 ngành × 10 công ty) giai đoạn 2015–2025  
**Nguồn dữ liệu**: [SEC.gov Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets)  
**Tham khảo chỉ số**: Vietcombank Securities (VCBS)

---

## Mục lục
1. [Setup & Import](#1-setup--import)
2. [Danh sách công ty mục tiêu](#2-danh-sách-công-ty-mục-tiêu)
3. [Mapping Ticker → CIK từ SEC.gov](#3-mapping-ticker--cik-từ-secgov)
4. [Bước 1: Cắt tỉa dữ liệu thô (Filter)](#4-bước-1-cắt-tỉa-dữ-liệu-thô-filter)
5. [Bước 2: Gộp các quý thành CSV (Merge)](#5-bước-2-gộp-các-quý-thành-csv-merge)
6. [Bước 3: EDA — Khảo sát dữ liệu](#6-bước-3-eda--khảo-sát-dữ-liệu)
7. [Bước 4: Làm sạch dữ liệu (Cleaning)](#7-bước-4-làm-sạch-dữ-liệu-cleaning)
8. [Tổng kết](#8-tổng-kết)

## 1. Setup & Import

In [1]:
import os
import sys
import json
import csv
import glob
import time
import urllib.request
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dir:      {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: c:\Users\phucb\Documents\Code\Project-2
Raw dir:      c:\Users\phucb\Documents\Code\Project-2\data\raw
Processed dir: c:\Users\phucb\Documents\Code\Project-2\data\processed


## 2. Danh sách công ty mục tiêu

50 công ty lớn chia thành 5 ngành, mỗi ngành 10 công ty:

| # | Ngành | Tickers |
|---|---|---|
| 1 | **Technology** | AAPL, MSFT, NVDA, INTC, AMD, CSCO, ORCL, IBM, CRM, ADBE |
| 2 | **Financial Services** | JPM, BAC, WFC, C, GS, MS, AXP, BLK, SCHW, USB |
| 3 | **Healthcare** | JNJ, PFE, MRK, ABBV, BMY, AMGN, GILD, MDT, CVS, UNH |
| 4 | **Consumer / Retail** | WMT, COST, HD, LOW, MCD, KO, PEP, SBUX, TGT, NKE |
| 5 | **Energy & Industrials** | XOM, CVX, COP, SLB, EOG, CAT, DE, GE, HON, UPS |

In [2]:
TARGET_TICKERS = [
    # Technology (10)
    "AAPL", "MSFT", "NVDA", "INTC", "AMD", "CSCO", "ORCL", "IBM", "CRM", "ADBE",
    # Financial Services (10)
    "JPM", "BAC", "WFC", "C", "GS", "MS", "AXP", "BLK", "SCHW", "USB",
    # Healthcare (10)
    "JNJ", "PFE", "MRK", "ABBV", "BMY", "AMGN", "GILD", "MDT", "CVS", "UNH",
    # Consumer / Retail (10)
    "WMT", "COST", "HD", "LOW", "MCD", "KO", "PEP", "SBUX", "TGT", "NKE",
    # Energy & Industrials (10)
    "XOM", "CVX", "COP", "SLB", "EOG", "CAT", "DE", "GE", "HON", "UPS",
]

SECTOR_MAP = {}
sectors = ["Technology", "Financial Services", "Healthcare", "Consumer / Retail", "Energy & Industrials"]
for i, sector in enumerate(sectors):
    for ticker in TARGET_TICKERS[i*10:(i+1)*10]:
        SECTOR_MAP[ticker] = sector

print(f"Tổng số ticker: {len(TARGET_TICKERS)}")
print(f"Số ngành: {len(sectors)}")

Tổng số ticker: 50
Số ngành: 5


## 3. Mapping Ticker → CIK từ SEC.gov

SEC sử dụng mã **CIK (Central Index Key)** để nhận diện công ty, không phải Ticker.  
Ta cần tải mapping từ SEC.gov để chuyển đổi.

In [3]:
def fetch_ticker_to_cik():
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": "ProjectSEC DataAnalysis (contact@example.com)"}
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read().decode())
    return {v["ticker"].upper(): str(v["cik_str"]) for v in data.values()}

ticker_to_cik = fetch_ticker_to_cik()
print(f"Tổng số ticker trong SEC: {len(ticker_to_cik):,}")

# Lấy CIK cho 50 công ty mục tiêu
target_ciks = set()
cik_to_ticker = {}
missing = []

for ticker in TARGET_TICKERS:
    cik = ticker_to_cik.get(ticker)
    if cik:
        target_ciks.add(cik)
        cik_to_ticker[cik] = ticker
    else:
        missing.append(ticker)

if missing:
    print(f"⚠️ Không tìm thấy CIK cho: {missing}")

# Hiển thị mapping
mapping_df = pd.DataFrame([
    {"Ticker": t, "CIK": ticker_to_cik.get(t, "N/A"), "Sector": SECTOR_MAP[t]}
    for t in TARGET_TICKERS
])
print(f"\nSố CIK mục tiêu: {len(target_ciks)}")
mapping_df

Tổng số ticker trong SEC: 10,365

Số CIK mục tiêu: 50


,Ticker,CIK,Sector
0,AAPL,320193,Technology
1,MSFT,789019,Technology
2,NVDA,1045810,Technology
3,INTC,50863,Technology
4,AMD,2488,Technology
5,CSCO,858877,Technology
6,ORCL,1341439,Technology
7,IBM,51143,Technology
8,CRM,1108524,Technology
9,ADBE,796343,Technology


## 4. Bước 1: Cắt tỉa dữ liệu thô (Filter)

Dữ liệu SEC gốc rất lớn (mỗi file `num.txt` khoảng 300-500MB/quý).  
Ta chỉ giữ lại submissions của 50 công ty mục tiêu.

**Quy trình:**
1. Lọc `sub.txt` theo cột `cik` → thu được danh sách `adsh` (mã báo cáo) hợp lệ
2. Lọc `num.txt` và `pre.txt` theo cột `adsh`
3. Ghi đè file gốc bằng file đã lọc  
4. Đọc/ghi theo từng dòng (line-by-line) để tiết kiệm RAM

In [4]:
def get_col_idx(header_line, col_name):
    cols = header_line.rstrip("\r\n").split("\t")
    try:
        return cols.index(col_name)
    except ValueError:
        return -1

def filter_sub(sub_path, target_ciks):
    tmp = sub_path + ".tmp"
    valid_adshs = set()
    kept = total = 0
    with open(sub_path, "r", encoding="utf-8", errors="replace") as fin, \
         open(tmp, "w", encoding="utf-8") as fout:
        header = fin.readline()
        fout.write(header)
        cik_idx = get_col_idx(header, "cik")
        adsh_idx = get_col_idx(header, "adsh")
        if cik_idx < 0 or adsh_idx < 0:
            os.remove(tmp)
            return set()
        for line in fin:
            total += 1
            parts = line.split("\t")
            if len(parts) > max(cik_idx, adsh_idx):
                if parts[cik_idx].strip() in target_ciks:
                    fout.write(line)
                    valid_adshs.add(parts[adsh_idx].strip())
                    kept += 1
    os.replace(tmp, sub_path)
    return valid_adshs, kept, total

def filter_by_adsh(file_path, valid_adshs):
    if not os.path.exists(file_path):
        return 0, 0
    tmp = file_path + ".tmp"
    kept = total = 0
    with open(file_path, "r", encoding="utf-8", errors="replace") as fin, \
         open(tmp, "w", encoding="utf-8") as fout:
        header = fin.readline()
        fout.write(header)
        adsh_idx = get_col_idx(header, "adsh")
        if adsh_idx < 0:
            os.remove(tmp)
            return 0, 0
        for line in fin:
            total += 1
            parts = line.split("\t")
            if len(parts) > adsh_idx and parts[adsh_idx].strip() in valid_adshs:
                fout.write(line)
                kept += 1
    os.replace(tmp, file_path)
    return kept, total

print("Đã định nghĩa các hàm filter.")

Đã định nghĩa các hàm filter.


In [5]:
# Tìm các thư mục quý
quarter_dirs = sorted([
    os.path.join(RAW_DIR, d)
    for d in os.listdir(RAW_DIR)
    if os.path.isdir(os.path.join(RAW_DIR, d))
])
print(f"Tìm thấy {len(quarter_dirs)} thư mục quý:")
print([os.path.basename(d) for d in quarter_dirs])

Tìm thấy 44 thư mục quý:
['2015q1', '2015q2', '2015q3', '2015q4', '2016q1', '2016q2', '2016q3', '2016q4', '2017q1', '2017q2', '2017q3', '2017q4', '2018q1', '2018q2', '2018q3', '2018q4', '2019q1', '2019q2', '2019q3', '2019q4', '2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1', '2022q2', '2022q3', '2022q4', '2023q1', '2023q2', '2023q3', '2023q4', '2024q1', '2024q2', '2024q3', '2024q4', '2025q1', '2025q2', '2025q3', '2025q4']


In [6]:
# Thực hiện lọc từng quý
FILTER_MARKER = ".filtered_done"
results = []
start = time.time()

for qdir in quarter_dirs:
    name = os.path.basename(qdir)
    marker = os.path.join(qdir, FILTER_MARKER)
    sub_path = os.path.join(qdir, "sub.txt")
    
    if os.path.exists(marker):
        print(f"[SKIP] {name} — đã lọc trước đó")
        results.append({"quarter": name, "status": "skipped"})
        continue
    
    if not os.path.exists(sub_path):
        print(f"[SKIP] {name} — không có sub.txt")
        continue
    
    # Filter sub.txt
    valid_adshs, sub_kept, sub_total = filter_sub(sub_path, target_ciks)
    
    # Filter num.txt, pre.txt
    num_kept, num_total = filter_by_adsh(os.path.join(qdir, "num.txt"), valid_adshs)
    pre_kept, pre_total = filter_by_adsh(os.path.join(qdir, "pre.txt"), valid_adshs)
    
    # Đánh dấu đã lọc
    with open(marker, "w") as f:
        f.write(f"filtered_at={time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    results.append({
        "quarter": name,
        "status": "filtered",
        "sub": f"{sub_kept:,}/{sub_total:,}",
        "num": f"{num_kept:,}/{num_total:,}",
        "pre": f"{pre_kept:,}/{pre_total:,}",
        "submissions": len(valid_adshs),
    })
    print(f"[DONE] {name}: sub={sub_kept}/{sub_total}, num={num_kept}/{num_total}, pre={pre_kept}/{pre_total}")

elapsed = time.time() - start
print(f"\nThời gian lọc: {elapsed:.1f} giây")

# Hiển thị kết quả
pd.DataFrame(results)

[SKIP] 2015q1 — đã lọc trước đó
[SKIP] 2015q2 — đã lọc trước đó
[SKIP] 2015q3 — đã lọc trước đó
[SKIP] 2015q4 — đã lọc trước đó
[SKIP] 2016q1 — đã lọc trước đó
[SKIP] 2016q2 — đã lọc trước đó
[SKIP] 2016q3 — đã lọc trước đó
[SKIP] 2016q4 — đã lọc trước đó
[SKIP] 2017q1 — đã lọc trước đó
[SKIP] 2017q2 — đã lọc trước đó
[SKIP] 2017q3 — đã lọc trước đó
[SKIP] 2017q4 — đã lọc trước đó
[SKIP] 2018q1 — đã lọc trước đó
[SKIP] 2018q2 — đã lọc trước đó
[SKIP] 2018q3 — đã lọc trước đó
[SKIP] 2018q4 — đã lọc trước đó
[SKIP] 2019q1 — đã lọc trước đó
[SKIP] 2019q2 — đã lọc trước đó
[SKIP] 2019q3 — đã lọc trước đó
[SKIP] 2019q4 — đã lọc trước đó
[SKIP] 2020q1 — đã lọc trước đó
[SKIP] 2020q2 — đã lọc trước đó
[SKIP] 2020q3 — đã lọc trước đó
[SKIP] 2020q4 — đã lọc trước đó
[SKIP] 2021q1 — đã lọc trước đó
[SKIP] 2021q2 — đã lọc trước đó
[SKIP] 2021q3 — đã lọc trước đó
[SKIP] 2021q4 — đã lọc trước đó
[SKIP] 2022q1 — đã lọc trước đó
[SKIP] 2022q2 — đã lọc trước đó
[SKIP] 2022q3 — đã lọc trước đó
[SKIP] 2

,quarter,status
0,2015q1,skipped
1,2015q2,skipped
2,2015q3,skipped
3,2015q4,skipped
4,2016q1,skipped
5,2016q2,skipped
6,2016q3,skipped
7,2016q4,skipped
8,2017q1,skipped
9,2017q2,skipped


## 5. Bước 2: Gộp các quý thành CSV (Merge)

Gộp tất cả các file `sub.txt`, `num.txt`, `pre.txt` đã lọc thành **3 file CSV duy nhất** trong `data/processed/`.  
Thêm cột `quarter` để biết dòng thuộc quý nào.

In [7]:
def merge_all_quarters(quarter_dirs, data_file, out_name):
    out_path = os.path.join(PROCESSED_DIR, out_name)
    total_rows = 0
    header_written = False
    
    with open(out_path, "w", encoding="utf-8", newline="") as fout:
        writer = None
        for qdir in quarter_dirs:
            quarter_name = os.path.basename(qdir)
            src = os.path.join(qdir, data_file)
            if not os.path.exists(src):
                continue
            with open(src, "r", encoding="utf-8", errors="replace") as fin:
                header_line = fin.readline().rstrip("\r\n")
                cols = header_line.split("\t")
                if not header_written:
                    writer = csv.writer(fout)
                    writer.writerow(["quarter"] + cols)
                    header_written = True
                for line in fin:
                    row = line.rstrip("\r\n").split("\t")
                    writer.writerow([quarter_name] + row)
                    total_rows += 1
    
    size_mb = os.path.getsize(out_path) / 1024 / 1024
    print(f"  {out_name}: {total_rows:,} dòng, {size_mb:.2f} MB")
    return total_rows, size_mb

print("Gộp dữ liệu các quý...\n")
merge_results = {}
for data_file, out_name in [("sub.txt", "sub_all.csv"), ("num.txt", "num_all.csv"), ("pre.txt", "pre_all.csv")]:
    rows, size = merge_all_quarters(quarter_dirs, data_file, out_name)
    merge_results[out_name] = {"rows": rows, "size_mb": size}

print("\n✅ Merge hoàn tất!")
pd.DataFrame(merge_results).T

Gộp dữ liệu các quý...

  sub_all.csv: 2,199 dòng, 0.61 MB
  num_all.csv: 1,529,847 dòng, 234.05 MB
  pre_all.csv: 271,371 dòng, 34.40 MB

✅ Merge hoàn tất!


,rows,size_mb
sub_all.csv,"2,199.00",0.61
num_all.csv,"1,529,847.00",234.05
pre_all.csv,"271,371.00",34.40


### Xóa file `.zip` gốc (giải phóng dung lượng)

In [8]:
zip_files = glob.glob(os.path.join(RAW_DIR, "*.zip"))
if zip_files:
    total_size = 0
    for zf in zip_files:
        size = os.path.getsize(zf)
        total_size += size
        os.remove(zf)
        print(f"  🗑️ {os.path.basename(zf)} ({size / 1024 / 1024:.1f} MB)")
    print(f"\n✅ Đã xóa {len(zip_files)} file .zip, giải phóng {total_size / 1024 / 1024:.1f} MB")
else:
    print("Không có file .zip nào để xóa (đã xóa trước đó).")

Không có file .zip nào để xóa (đã xóa trước đó).


## 6. Bước 3: EDA — Khảo sát dữ liệu

Đọc 3 file CSV đã gộp và phân tích:
- Kích thước, kiểu dữ liệu
- Missing values
- Duplicates
- Phân bố giá trị
- Outliers

In [9]:
# Load dữ liệu
sub = pd.read_csv(os.path.join(PROCESSED_DIR, "sub_all.csv"))
num = pd.read_csv(os.path.join(PROCESSED_DIR, "num_all.csv"))
pre = pd.read_csv(os.path.join(PROCESSED_DIR, "pre_all.csv"))

print("📊 Kích thước dữ liệu:")
print(f"  sub: {sub.shape[0]:>10,} dòng × {sub.shape[1]} cột  ({sub.memory_usage(deep=True).sum()/1024/1024:.1f} MB)")
print(f"  num: {num.shape[0]:>10,} dòng × {num.shape[1]} cột  ({num.memory_usage(deep=True).sum()/1024/1024:.1f} MB)")
print(f"  pre: {pre.shape[0]:>10,} dòng × {pre.shape[1]} cột  ({pre.memory_usage(deep=True).sum()/1024/1024:.1f} MB)")

📊 Kích thước dữ liệu:
  sub:      2,199 dòng × 37 cột  (3.0 MB)
  num:  1,529,847 dòng × 11 cột  (731.0 MB)
  pre:    271,371 dòng × 11 cột  (127.0 MB)


### 6.1 EDA: SUB (Submissions)

In [10]:
print("=== DTYPES ===")
print(sub.dtypes.to_string())
print(f"\n=== MISSING VALUES ===")
missing = sub.isnull().sum()
missing_pct = (sub.isnull().sum() / len(sub) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct(%)": missing_pct, "dtype": sub.dtypes})
print(missing_df[missing_df["count"] > 0].sort_values("pct(%)", ascending=False).to_string())

print(f"\n=== KEY STATS ===")
print(f"  Unique CIK (companies): {sub['cik'].nunique()}")
print(f"  Unique company names:   {sub['name'].nunique()}")
print(f"  Duplicate adsh:         {sub.duplicated(subset=['adsh']).sum()}")
print(f"  prevrpt=1 (amended):    {(sub['prevrpt']==1).sum()}")
print(f"  Fiscal year range:      {sub['fy'].min():.0f} — {sub['fy'].max():.0f}")
print(f"  Period range:           {sub['period'].min()} — {sub['period'].max()}")

print(f"\n=== FORM TYPES ===")
print(sub["form"].value_counts().to_string())

print(f"\n=== SUBMISSIONS PER QUARTER ===")
print(sub["quarter"].value_counts().sort_index().to_string())

=== DTYPES ===
quarter           str
adsh              str
cik             int64
name              str
sic             int64
countryba         str
stprba            str
cityba            str
zipba             str
bas1              str
bas2              str
baph              str
countryma         str
stprma            str
cityma            str
zipma             str
mas1              str
mas2              str
countryinc        str
stprinc           str
ein             int64
former            str
changed       float64
afs               str
wksi            int64
fye             int64
form              str
period          int64
fy            float64
fp                str
filed           int64
accepted          str
prevrpt         int64
detail          int64
instance          str
nciks           int64
aciks         float64

=== MISSING VALUES ===
            count  pct(%)    dtype
aciks        2199  100.00  float64
mas2         1769   80.45      str
bas2         1740   79.13      str
changed

### 6.2 EDA: NUM (Numbers)

In [11]:
print("=== MISSING VALUES ===")
missing = num.isnull().sum()
missing_pct = (num.isnull().sum() / len(num) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct(%)": missing_pct})
print(missing_df[missing_df["count"] > 0].sort_values("pct(%)", ascending=False).to_string())

print(f"\n=== VALUE STATISTICS ===")
print(num["value"].describe().to_string())

print(f"\n=== OUTLIERS (IQR) ===")
q1 = num["value"].quantile(0.25)
q3 = num["value"].quantile(0.75)
iqr = q3 - q1
outliers = num[(num["value"] < q1 - 1.5*iqr) | (num["value"] > q3 + 1.5*iqr)]
print(f"  Q1={q1:,.0f}, Q3={q3:,.0f}, IQR={iqr:,.0f}")
print(f"  Outliers: {len(outliers):,} / {len(num):,} ({len(outliers)/len(num)*100:.1f}%)")

print(f"\n=== NEGATIVE & ZERO ===")
print(f"  Negative: {(num['value']<0).sum():,} ({(num['value']<0).sum()/len(num)*100:.1f}%)")
print(f"  Zero:     {(num['value']==0).sum():,} ({(num['value']==0).sum()/len(num)*100:.1f}%)")

print(f"\n=== UOM (Unit of Measure) ===")
print(num["uom"].value_counts().to_string())

print(f"\n=== SEGMENTS (segment-level data) ===")
seg_filled = num["segments"].notna().sum()
print(f"  Có segment: {seg_filled:,} ({seg_filled/len(num)*100:.1f}%)")
print(f"  Consolidated (null): {num['segments'].isna().sum():,} ({num['segments'].isna().sum()/len(num)*100:.1f}%)")

print(f"\n=== TOP 15 TAGS ===")
for i, (tag, cnt) in enumerate(num["tag"].value_counts().head(15).items(), 1):
    print(f"  {i:>2}. {tag:60s} {cnt:>8,}")

=== MISSING VALUES ===
            count  pct(%)
footnote  1525301   99.70
coreg     1502279   98.20
segments   633267   41.39
value        2919    0.19

=== VALUE STATISTICS ===
count               1,526,928.00
mean           36,293,348,377.16
std         6,198,357,929,860.72
min       -24,121,801,000,000.00
25%                 1,000,000.00
50%               538,000,000.00
75%             4,782,000,000.00
max     2,442,676,580,000,000.00

=== OUTLIERS (IQR) ===
  Q1=1,000,000, Q3=4,782,000,000, IQR=4,781,000,000
  Outliers: 262,130 / 1,529,847 (17.1%)

=== NEGATIVE & ZERO ===
  Negative: 201,147 (13.1%)
  Zero:     134,231 (8.8%)

=== UOM (Unit of Measure) ===
uom
USD       1473092
shares      52190
pure         4481
EUR            48
JPY            26
GBP            10

=== SEGMENTS (segment-level data) ===
  Có segment: 896,580 (58.6%)
  Consolidated (null): 633,267 (41.4%)

=== TOP 15 TAGS ===
   1. RevenueFromContractWithCustomerExcludingAssessedTax            78,178
   2. Revenue

### 6.3 EDA: PRE (Presentation)

In [12]:
print("=== MISSING VALUES ===")
missing = pre.isnull().sum()
print(pd.DataFrame({"count": missing, "pct(%)": (missing/len(pre)*100).round(2)})[missing > 0].to_string())

print(f"\n=== STATEMENT TYPES (stmt) ===")
print(pre["stmt"].value_counts().to_string())

print(f"\n=== DUPLICATES (adsh, report, line) ===")
print(f"  {pre.duplicated(subset=['adsh','report','line']).sum()} duplicates")

=== MISSING VALUES ===
      count  pct(%)
stmt     50    0.02

=== STATEMENT TYPES (stmt) ===
stmt
BS    89012
CF    80197
IS    51445
EQ    27356
CI    22504
UN      645
SI      162

=== DUPLICATES (adsh, report, line) ===
  0 duplicates


## 7. Bước 4: Làm sạch dữ liệu (Cleaning)

Dựa trên kết quả EDA, các vấn đề cần xử lý:

| # | Vấn đề | Xử lý |
|---|---|---|
| 1 | `SUB`: 24 báo cáo sửa đổi (prevrpt=1) | Loại bỏ, chỉ giữ báo cáo gốc |
| 2 | `SUB`: 8 cột metadata thừa | Drop (bas2, mas2, baph, former, changed, aciks, instance, detail) |
| 3 | `SUB`: period/filed ở dạng int | Chuyển sang datetime |
| 4 | `NUM`: 2,919 dòng value=null | Loại bỏ |
| 5 | `NUM`: 58.6% dữ liệu segment-level | Loại, chỉ giữ consolidated |
| 6 | `NUM`: cột coreg/footnote/segments | Drop (hầu hết null hoặc không cần) |
| 7 | `PRE`: 857 dòng statement type UN/SI | Loại, chỉ giữ BS/IS/CF/EQ/CI |

### 7.1 Cleaning: SUB

In [13]:
print(f"Trước: {len(sub):,} dòng, {sub.shape[1]} cột")

# 1. Loại báo cáo sửa đổi
sub_clean = sub[sub["prevrpt"] == 0].copy()
print(f"  - Drop prevrpt=1: loại {len(sub) - len(sub_clean)} dòng")

# 2. Drop duplicate adsh
before = len(sub_clean)
sub_clean = sub_clean.drop_duplicates(subset=["adsh"], keep="first")
print(f"  - Drop duplicate adsh: loại {before - len(sub_clean)} dòng")

# 3. Chuẩn hóa period, filed -> datetime
sub_clean["period"] = pd.to_datetime(sub_clean["period"].astype(str), format="%Y%m%d", errors="coerce")
sub_clean["filed"] = pd.to_datetime(sub_clean["filed"].astype(str), format="%Y%m%d", errors="coerce")
print(f"  - Chuyển period/filed sang datetime")

# 4. Drop cột metadata thừa
drop_cols = ["bas2", "mas2", "baph", "former", "changed", "aciks", "instance", "detail"]
existing = [c for c in drop_cols if c in sub_clean.columns]
sub_clean = sub_clean.drop(columns=existing)
print(f"  - Drop {len(existing)} cột: {existing}")

# 5. Chuẩn hóa tên
sub_clean["name"] = sub_clean["name"].str.strip().str.upper()

print(f"\n✅ Sau: {len(sub_clean):,} dòng, {sub_clean.shape[1]} cột")
sub_clean.head()

Trước: 2,199 dòng, 37 cột
  - Drop prevrpt=1: loại 24 dòng
  - Drop duplicate adsh: loại 0 dòng
  - Chuyển period/filed sang datetime
  - Drop 8 cột: ['bas2', 'mas2', 'baph', 'former', 'changed', 'aciks', 'instance', 'detail']

✅ Sau: 2,175 dòng, 29 cột


,quarter,adsh,cik,name,sic,countryba,stprba,cityba,zipba,bas1,countryma,stprma,cityma,zipma,mas1,countryinc,stprinc,ein,afs,wksi,fye,form,period,fy,fp,filed,accepted,prevrpt,nciks
0,2015q1,0000077476-15-000012,77476,PEPSICO INC,2080,US,NY,PURCHASE,10577,700 ANDERSON HILL RD,US,NY,PURCHASE,10577-1444,700 ANDERSON HILL ROAD,US,NC,131584302,1-LAF,1,1231,10-K,2014-12-31,"2,014.00",FY,2015-02-12,2015-02-12 17:02:00.0,0,1
1,2015q1,0000829224-15-000006,829224,STARBUCKS CORP,5810,US,WA,SEATTLE,98124-1067,P O BOX 34067,US,WA,SEATTLE,98134,2401 UTAH AVENUE SOUTH,US,WA,911325671,1-LAF,0,930,10-Q,2014-12-31,"2,015.00",Q1,2015-01-27,2015-01-27 16:09:00.0,0,1
2,2015q1,0000060667-15-000057,60667,LOWES COMPANIES INC,5211,US,NC,MOORESVILLE,28117,1000 LOWE'S BLVD.,US,NC,MOORESVILLE,28115,P.O. BOX 1000,US,NC,560578072,1-LAF,1,131,10-K,2015-01-31,"2,014.00",FY,2015-03-31,2015-03-31 15:10:00.0,0,1
3,2015q1,0000909832-15-000003,909832,COSTCO WHOLESALE CORP /NEW,5331,US,WA,ISSAQUAH,98027-,999 LAKE DRIVE,US,WA,ISSAQUAH,98027,999 LAKE DRIVE,US,WA,911223280,1-LAF,0,831,10-Q,2015-01-31,"2,015.00",Q2,2015-03-11,2015-03-11 17:24:00.0,0,1
4,2015q1,0000731766-15-000007,731766,UNITEDHEALTH GROUP INC,6324,US,MN,MINNEAPOLIS,55343,UNITEDHEALTH GROUP CENTER,US,MN,MINNETONKA,55343,9900 BREN ROAD EAST,US,MN,411321939,1-LAF,1,1231,10-K,2014-12-31,"2,014.00",FY,2015-02-10,2015-02-10 16:45:00.0,0,1


### 7.2 Cleaning: NUM

In [14]:
print(f"Trước: {len(num):,} dòng, {num.shape[1]} cột")

# 1. Drop value = null
num_clean = num.dropna(subset=["value"]).copy()
print(f"  - Drop value=null: loại {len(num) - len(num_clean):,} dòng")

# 2. Drop duplicates
before = len(num_clean)
key_cols = ["adsh", "tag", "version", "ddate", "qtrs", "uom", "segments", "coreg"]
num_clean = num_clean.drop_duplicates(subset=key_cols, keep="first")
print(f"  - Drop duplicates: loại {before - len(num_clean):,} dòng")

# 3. Chỉ giữ consolidated (segments = null)
before = len(num_clean)
num_clean = num_clean[num_clean["segments"].isna()].copy()
print(f"  - Filter consolidated (segments=null): loại {before - len(num_clean):,} dòng segment-level")

# 4. Drop cột thừa
drop_cols = ["coreg", "footnote", "segments"]
existing = [c for c in drop_cols if c in num_clean.columns]
num_clean = num_clean.drop(columns=existing)
print(f"  - Drop cột: {existing}")

# 5. Chuẩn hóa ddate
num_clean["ddate"] = pd.to_datetime(num_clean["ddate"].astype(str), format="%Y%m%d", errors="coerce")

print(f"\n✅ Sau: {len(num_clean):,} dòng, {num_clean.shape[1]} cột")
num_clean.head()

Trước: 1,529,847 dòng, 11 cột
  - Drop value=null: loại 2,919 dòng
  - Drop duplicates: loại 0 dòng
  - Filter consolidated (segments=null): loại 896,215 dòng segment-level
  - Drop cột: ['coreg', 'footnote', 'segments']

✅ Sau: 630,713 dòng, 8 cột


,quarter,adsh,tag,version,ddate,qtrs,uom,value
11,2015q1,0001193125-15-023732,PaymentsForProceedsFromOtherInvestingActivities,us-gaap/2014,2012-09-30,4,USD,"48,000,000.00"
16,2015q1,0000310158-15-000005,EquityMethodInvestmentDividendsOrDistributions,us-gaap/2014,2014-12-31,4,USD,"185,000,000.00"
18,2015q1,0001193125-15-098586,PaymentsOfDividendsCommonStock,us-gaap/2014,2015-02-28,3,USD,"1,600,000,000.00"
20,2015q1,0001193125-15-071980,LaborAndRelatedExpense,us-gaap/2014,2013-12-31,4,USD,"16,277,000,000.00"
21,2015q1,0001193125-15-098586,IncreaseDecreaseInPrepaidDeferredExpenseAndOth...,us-gaap/2014,2014-02-28,3,USD,"-181,000,000.00"


### 7.3 Cleaning: PRE

In [15]:
print(f"Trước: {len(pre):,} dòng, {pre.shape[1]} cột")

# 1. Drop duplicates
before = len(pre)
pre_clean = pre.drop_duplicates(subset=["adsh", "report", "line"], keep="first").copy()
print(f"  - Drop duplicates: loại {before - len(pre_clean)} dòng")

# 2. Chỉ giữ statement chính
valid_stmts = ["BS", "IS", "CF", "EQ", "CI"]
before = len(pre_clean)
pre_clean = pre_clean[pre_clean["stmt"].isin(valid_stmts)]
print(f"  - Filter valid statements {valid_stmts}: loại {before - len(pre_clean)} dòng")

print(f"\n✅ Sau: {len(pre_clean):,} dòng, {pre_clean.shape[1]} cột")
pre_clean.head()

Trước: 271,371 dòng, 11 cột
  - Drop duplicates: loại 0 dòng
  - Filter valid statements ['BS', 'IS', 'CF', 'EQ', 'CI']: loại 857 dòng

✅ Sau: 270,514 dòng, 11 cột


,quarter,adsh,report,line,stmt,inpth,rfile,tag,version,plabel,negating
0,2015q1,0000014272-15-000055,2,1,IS,0,H,SalesRevenueGoodsNet,us-gaap/2013,Net product sales,0
1,2015q1,0000014272-15-000055,2,2,IS,0,H,OtherSalesRevenueNet,us-gaap/2013,Alliance and other revenues,0
2,2015q1,0000014272-15-000055,2,3,IS,0,H,SalesRevenueNet,us-gaap/2013,Total Revenues,0
3,2015q1,0000014272-15-000055,2,4,IS,0,H,CostOfGoodsSold,us-gaap/2013,Cost of products sold,0
4,2015q1,0000014272-15-000055,2,5,IS,0,H,SellingGeneralAndAdministrativeExpense,us-gaap/2013,"Marketing, selling and administrative",0


## 8. Tổng kết

### Lưu dữ liệu sạch

In [16]:
# Lưu file sạch
sub_clean.to_csv(os.path.join(PROCESSED_DIR, "sub_clean.csv"), index=False)
num_clean.to_csv(os.path.join(PROCESSED_DIR, "num_clean.csv"), index=False)
pre_clean.to_csv(os.path.join(PROCESSED_DIR, "pre_clean.csv"), index=False)

print("✅ Đã lưu dữ liệu sạch vào data/processed/\n")

# Tổng kết
summary = pd.DataFrame({
    "Dataset": ["sub", "num", "pre"],
    "Trước (dòng)": [f"{len(sub):,}", f"{len(num):,}", f"{len(pre):,}"],
    "Sau (dòng)": [f"{len(sub_clean):,}", f"{len(num_clean):,}", f"{len(pre_clean):,}"],
    "Giảm (%)": [
        f"{(1-len(sub_clean)/len(sub))*100:.1f}%",
        f"{(1-len(num_clean)/len(num))*100:.1f}%",
        f"{(1-len(pre_clean)/len(pre))*100:.1f}%",
    ],
})
print("📊 So sánh trước/sau:\n")
print(summary.to_string(index=False))

# Dung lượng file
print("\n📁 Dung lượng file output:")
for f in ["sub_clean.csv", "num_clean.csv", "pre_clean.csv"]:
    path = os.path.join(PROCESSED_DIR, f)
    size = os.path.getsize(path) / 1024 / 1024
    print(f"  {f:20s} {size:.2f} MB")

✅ Đã lưu dữ liệu sạch vào data/processed/

📊 So sánh trước/sau:

Dataset Trước (dòng) Sau (dòng) Giảm (%)
    sub        2,199      2,175     1.1%
    num    1,529,847    630,713    58.8%
    pre      271,371    270,514     0.3%

📁 Dung lượng file output:
  sub_clean.csv        0.49 MB
  num_clean.csv        67.54 MB
  pre_clean.csv        34.28 MB


### Các bước tiếp theo

- **Schema PostgreSQL**: Thiết kế bảng và import dữ liệu sạch
- **Duckle-ETL**: Xây dựng pipeline tự động
- **PowerBI**: Tạo dashboard phân tích các chỉ số tài chính (tham khảo VCBS)

---
*Notebook này có thể chạy lại bất kỳ lúc nào khi có thêm dữ liệu mới.*